# Part 1: Workload Forecasting with LSTM
**Project:** ML-Based Predictive Load Balancing and Auto-Scaling in Cloud Environments

Steps: convert NASA HTTP logs to requests per 5 minutes, train an LSTM to predict the next interval, compare with Naive and Moving Average baselines, and save `predictions.csv` (for CloudSim Plus) and the model (for the API).

Use Runtime > Change runtime type > T4 GPU in Colab (CPU also works but is slower).

## 1. Setup

In [ ]:
import re, os, gzip, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

INTERVAL = '5min'   # interval size (same as one CloudSim step)
WINDOW   = 12       # use last 12 values = 1 hour
os.makedirs('outputs', exist_ok=True)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 2. Get the dataset
NASA Kennedy Space Center HTTP logs (July 1995), from the Internet Traffic Archive.
If the FTP download fails, download `NASA_access_log_Jul95.gz` yourself and upload it when prompted.

In [ ]:
FNAME = 'NASA_access_log_Jul95.gz'
if not os.path.exists(FNAME):
    !wget -q --timeout=60 ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz

if not os.path.exists(FNAME) or os.path.getsize(FNAME) < 1_000_000:
    print('Download failed, please upload NASA_access_log_Jul95.gz')
    from google.colab import files
    up = files.upload()
    FNAME = list(up.keys())[0]

print(FNAME, round(os.path.getsize(FNAME)/1e6, 1), 'MB')

## 3. Convert logs to requests per 5 minutes

In [ ]:
ts_re = re.compile(r'\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2})')
stamps = []
opener = gzip.open if FNAME.endswith('.gz') else open
with opener(FNAME, 'rt', encoding='latin-1', errors='ignore') as f:
    for line in f:
        m = ts_re.search(line)
        if m:
            stamps.append(m.group(1))

ts = pd.to_datetime(pd.Series(stamps), format='%d/%b/%Y:%H:%M:%S', errors='coerce').dropna()
print('Total requests parsed:', len(ts))

series = ts.dt.floor(INTERVAL).value_counts().sort_index()
full_idx = pd.date_range(series.index.min(), series.index.max(), freq=INTERVAL)
series = series.reindex(full_idx, fill_value=0).astype(float)
series.name = 'requests'
print('Intervals:', len(series))
series.describe()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7))
series.plot(ax=ax[0], lw=0.6, color='steelblue')
ax[0].set_title('NASA HTTP workload: requests per 5 minutes (July 1995)')
ax[0].set_ylabel('Requests')
series.iloc[:288*3].plot(ax=ax[1], lw=1, color='darkorange')
ax[1].set_title('Zoom: first 3 days (daily cycle)')
ax[1].set_ylabel('Requests')
plt.tight_layout(); plt.savefig('outputs/fig1_workload.png', dpi=200); plt.show()

## 4. Train / validation / test split (chronological) + scaling

In [ ]:
values = series.values.reshape(-1, 1)
n = len(values)
n_train, n_val = int(n*0.70), int(n*0.15)

train = values[:n_train]
val   = values[n_train:n_train+n_val]
test  = values[n_train+n_val:]

scaler = MinMaxScaler()
scaler.fit(train)                      # fit on training data only
train_s, val_s, test_s = scaler.transform(train), scaler.transform(val), scaler.transform(test)

def make_windows(arr, w):
    X, y = [], []
    for i in range(w, len(arr)):
        X.append(arr[i-w:i]); y.append(arr[i])
    return np.array(X), np.array(y)

# add the last WINDOW points before val/test so no interval is skipped
X_train, y_train = make_windows(train_s, WINDOW)
X_val,   y_val   = make_windows(np.vstack([train_s[-WINDOW:], val_s]), WINDOW)
X_test,  y_test  = make_windows(np.vstack([val_s[-WINDOW:], test_s]), WINDOW)
print('Train', X_train.shape, '| Val', X_val.shape, '| Test', X_test.shape)

## 5. LSTM model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(WINDOW, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

cb = [keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
      keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, monitor='val_loss')]
hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=60, batch_size=64, callbacks=cb, verbose=1)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist.history['loss'], label='Train loss')
plt.plot(hist.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.title('LSTM training curve')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('outputs/fig2_training_curve.png', dpi=200); plt.show()

## 6. Predictions + baselines

In [ ]:
inv = lambda a: scaler.inverse_transform(a.reshape(-1, 1)).ravel()

actual   = inv(y_test)
lstm_pred = np.clip(inv(model.predict(X_test, verbose=0)), 0, None)

# Baseline 1: Naive (next interval = last interval)
naive_pred = inv(X_test[:, -1, 0])
# Baseline 2: Moving average of the last hour
ma_pred = inv(X_test[:, :, 0].mean(axis=1))

def metrics(y, p):
    mae  = mean_absolute_error(y, p)
    rmse = np.sqrt(mean_squared_error(y, p))
    mask = y > 0
    mape = np.mean(np.abs((y[mask]-p[mask]) / y[mask])) * 100
    return round(mae, 2), round(rmse, 2), round(mape, 2)

results = pd.DataFrame(
    [metrics(actual, naive_pred), metrics(actual, ma_pred), metrics(actual, lstm_pred)],
    index=['Naive (last value)', 'Moving Average (1h)', 'LSTM (proposed)'],
    columns=['MAE', 'RMSE', 'MAPE (%)'])
results.to_csv('outputs/forecast_metrics.csv')
results

In [ ]:
test_idx = series.index[n_train+n_val:]
k = 288*2   # show only 2 days
plt.figure(figsize=(14, 5))
plt.plot(test_idx[:k], actual[:k], label='Actual', color='black', lw=1.2)
plt.plot(test_idx[:k], lstm_pred[:k], label='LSTM', color='crimson', lw=1)
plt.plot(test_idx[:k], ma_pred[:k], label='Moving Avg', color='seagreen', lw=1, ls='--')
plt.title('Actual vs predicted workload (test set, first 2 days)')
plt.ylabel('Requests / 5 min'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('outputs/fig3_actual_vs_pred.png', dpi=200); plt.show()

ax = results[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4), rot=0, color=['#4c72b0', '#dd8452'])
ax.set_title('Forecast error comparison (lower is better)'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/fig4_error_comparison.png', dpi=200); plt.show()

## 7. Save outputs for CloudSim Plus and the API
- `predictions.csv`: one row for every 5 minute step of the test period, used by CloudSim.
- `lstm_model.keras`, `scaler.pkl` and `config.pkl`: used for the API.

In [ ]:
out = pd.DataFrame({
    'step': np.arange(len(actual)),
    'timestamp': test_idx.astype(str),
    'actual_requests': actual.round().astype(int),
    'lstm_pred': lstm_pred.round().astype(int),
    'ma_pred': ma_pred.round().astype(int),
    'naive_pred': naive_pred.round().astype(int),
})
out.to_csv('outputs/predictions.csv', index=False)

model.save('outputs/lstm_model.keras')
pickle.dump(scaler, open('outputs/scaler.pkl', 'wb'))
pickle.dump({'window': WINDOW, 'interval': INTERVAL}, open('outputs/config.pkl', 'wb'))
print(out.head()); print('Rows:', len(out))

In [ ]:
!zip -rq outputs.zip outputs
from google.colab import files
files.download('outputs.zip')